# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out dataset summary information
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Authors (@id):", [a['@id'] if isinstance(a, dict) and '@id' in a else str(a) for a in getattr(metadata, 'author', [])])
print("Published Date:", getattr(metadata, "datePublished", "N/A"))
print("Keywords:", getattr(metadata, "keywords", []))
print("Spatial Coverage:", getattr(metadata, "spatialCoverage", "N/A"))
print("Temporal Coverage:", getattr(metadata, "temporalCoverage", "N/A"))
print("License:", getattr(metadata, "license", "N/A"))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into Record Sets. We first inspect the record sets included in this dataset.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')} | description: {rs.get('description', 'N/A')}")

### Explore fields within a record set
Fields (columns) of each record set are referenced by their unique `@id`.

Let's inspect the fields of the first record set.

In [ ]:
# Examine fields (columns) for the first record set
first_record_set_id = record_sets[0]['@id'] if record_sets else None

fields = dataset.fields(record_set=first_record_set_id) if first_record_set_id else []
print(f"Fields for record set @id {first_record_set_id}:")
for field in fields:
    print(f"@id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract rows/records from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Record set @id: {rs_id}")
        print(f"Columns/Fields (@id): {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for record set @id: {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Identify a record set and a numeric field for EDA
if dataframes:
    # Select the first record set and a numeric column
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]
    numeric_fields = [col for col in df.columns if df[col].dtype in ["float64", "int64"]]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field if available
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframes from record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot a histogram of a numeric field and a bar plot grouped by a categorical variable (if available) from the first record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]
    # Plot histogram for first numeric field
    numeric_fields = [col for col in df.columns if df[col].dtype in ["float64", "int64"]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(7, 5))
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id} in record set {selected_rs_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

    # Bar plot for grouping
    group_fields = [col for col in df.columns if df[col].dtype == 'object']
    if numeric_fields and group_fields:
        group_field_id = group_fields[0]
        grp = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 6))
        sns.barplot(data=grp, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrates how to use `mlcroissant` to load FAIR^2 (Croissant) datasets directly from a schema URL.
- Data from each record set is referenced using their `@id`, ensuring reproducibility and clarity.
- Exploratory analysis and visualization revealed numerical and categorical distributions in the survey data.
- The dataset provides insights into knowledge adoption predictors for rangeland management, but users should be aware of potential biases and limitations as noted in the metadata.
